In [8]:
import pandas as pd
import numpy as np
import random

np.random.seed(42)
random.seed(42)

# 1. Load canonical merchant mapping
mapping_df = pd.read_csv('../data/raw/merchant_vat_mapping.csv')

locations = ['LONDON', 'MANCHESTER', 'BIRMINGHAM', 'LEEDS', 'BRISTOL', 'GLASGOW', 'ONLINE', 'UK']
prefixes = ['POS ', 'DD ', 'CARD ', 'PAYMENT TO ', 'SQ *', 'IZ *', '']
suffixes = [' LTD', ' PLC', ' UK', ' DIRECT', ' STORE', ' ONLINE', '']

synthetic_rows = []

# 2. Build synthetic transaction variations
for _, row in mapping_df.iterrows():
    base_merchant = row['merchant_keyword']
    vat_code = row['vat_code']
    
    for _ in range(150):
        pref = random.choice(prefixes)
        suff = random.choice(suffixes)
        loc = random.choice(locations) if random.random() > 0.4 else ''
        store_num = f" {random.randint(100, 9999)}" if random.random() > 0.3 else ''
        ref_code = f" REF-{random.randint(1000, 9999)}" if random.random() > 0.6 else ''
        
        desc = f"{pref}{base_merchant}{store_num} {loc}{suff}{ref_code}".strip()
        
        rand_val = random.random()
        if rand_val > 0.85:
            desc = desc.lower()
        elif rand_val > 0.70:
            desc = desc.title()
            
        synthetic_rows.append({
            'raw_description': desc,
            'vat_code': vat_code,
            'base_merchant': base_merchant
        })

df_synthetic = pd.DataFrame(synthetic_rows)
df_synthetic = df_synthetic.sample(frac=1.0, random_state=42).reset_index(drop=True)

# 3. Export to CSV
df_synthetic.to_csv('../data/raw/synthetic_bank_transactions.csv', index=False)
print(f"Generated {len(df_synthetic)} transaction records in data/raw/synthetic_bank_transactions.csv")

Generated 6150 transaction records in data/raw/synthetic_bank_transactions.csv


In [9]:
import glob
import pdfplumber
import pandas as pd

# Find all PDFs in data/raw/
pdf_files = glob.glob("../data/raw/*.pdf")
extracted_records = []

print(f"Found {len(pdf_files)} PDF statements in data/raw/")

# Extract text line-by-line from each PDF statement
for pdf_file in pdf_files:
    print(f"Processing: {pdf_file}")
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                for line in text.split("\n"):
                    line_clean = line.strip()
                    # Filter out empty or very short header lines
                    if len(line_clean) > 5:
                        extracted_records.append({
                            "source_file": pdf_file,
                            "raw_description": line_clean
                        })

# Store extracted lines as a DataFrame
pdf_df = pd.DataFrame(extracted_records)
print(f"\nExtracted {len(pdf_df)} raw text lines across all PDFs.")
pdf_df.head(10)

Found 12 PDF statements in data/raw/
Processing: ../data/raw\Amex_CC_Statement_Mar2025.pdf
Processing: ../data/raw\bank_statement_ds1.pdf
Processing: ../data/raw\bank_statement_ds2.pdf
Processing: ../data/raw\bank_statement_feb2026.pdf
Processing: ../data/raw\bank_statement_jan2026.pdf
Processing: ../data/raw\bank_statement_mar2026.pdf
Processing: ../data/raw\CapitalOne_CC_Statement_Mar2025.pdf
Processing: ../data/raw\Chase_CC_Statement_Mar2025.pdf
Processing: ../data/raw\restaurant_q1_bank_statement (1).pdf
Processing: ../data/raw\restaurant_q1_bank_statement (2).pdf
Processing: ../data/raw\restaurant_q1_bank_statement.pdf
Processing: ../data/raw\Restaurant_Statement_1_The_Bluegrass_Diner.pdf

Extracted 619 raw text lines across all PDFs.


,source_file,raw_description
0,../data/raw\Amex_CC_Statement_Mar2025.pdf,American Express Business Gold Card Statement
1,../data/raw\Amex_CC_Statement_Mar2025.pdf,Cardmember: Louisville BBQ Co. Closing Date: 0...
2,../data/raw\Amex_CC_Statement_Mar2025.pdf,Account Ending: 7-21009 Payment Due Date: 04/2...
3,../data/raw\Amex_CC_Statement_Mar2025.pdf,"Reward Points: 14,820 pts New Balance: $4,506.69"
4,../data/raw\Amex_CC_Statement_Mar2025.pdf,Charges — March 2025
5,../data/raw\Amex_CC_Statement_Mar2025.pdf,Date Description Appears On Statement As Amount
6,../data/raw\Amex_CC_Statement_Mar2025.pdf,03/03/2025 SPECTRUM INTERNET SPECTRUM INTERNET...
7,../data/raw\Amex_CC_Statement_Mar2025.pdf,03/04/2025 ULINE SHIPPING SUPPLY ULINE SHIPPIN...
8,../data/raw\Amex_CC_Statement_Mar2025.pdf,03/05/2025 DOORDASH FEE PLATFORM DOORDASH FEE ...
9,../data/raw\Amex_CC_Statement_Mar2025.pdf,03/07/2025 SAMS CLUB #6342 SAMS CLUB #6342 $40...


In [10]:
# Load synthetic generated transactions
synthetic_df = pd.read_csv("../data/raw/synthetic_bank_transactions.csv")

# Merge raw descriptions for final cleaning
full_dataset = pd.concat([
    synthetic_df[['raw_description', 'vat_code']],
    pdf_df[['raw_description']].assign(vat_code='standard_rate') # Default placeholder for unlabelled PDF lines
], ignore_index=True)

# Save master dataset for processing
full_dataset.to_csv("../data/raw/combined_transactions.csv", index=False)
print(f"Master dataset ready with {len(full_dataset)} total rows!")

Master dataset ready with 6769 total rows!
